# Findings 

- when an action leads to collide to the wall, it does nothing (stays in the same position before the action). 

- dot can be inside the wall 

- last state is discarded to match the action shape. 

In [1]:
import random
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

REPO_ROOT = Path.cwd().parents[1]          # examples/my_ac_video_jepa -> repo root
sys.path.insert(0, str(REPO_ROOT))

from eb_jepa.datasets.utils import init_data
from eb_jepa.datasets.two_rooms.utils import generate_wall_layouts

# torch.manual_seed(0); np.random.seed(0); random.seed(0)

In [2]:
print(REPO_ROOT)

/Users/hawardizayee/Desktop/AMI/eb_jepa/examples/my_ac_video_jepa


### `load_env_data_config`

In [3]:
import yaml 

# load_env_data_config(env_name, overrides)
env_name  = "two_rooms"
overrides = {"batch_size": 128}
#====================================
DATASETS_DIR = Path("/Users/hawardizayee/Desktop/AMI/eb_jepa/eb_jepa/datasets")
config_path = DATASETS_DIR / env_name / "data_config.yaml"  # data_config.yaml is hardcoded 

print(config_path)

with open(config_path) as f:
    base_config = yaml.safe_load(f)

base_config.update(overrides)

base_config

/Users/hawardizayee/Desktop/AMI/eb_jepa/eb_jepa/datasets/two_rooms/data_config.yaml


{'action_noise': 1,
 'action_angle_noise': 0.2,
 'action_step_mean': 1.0,
 'action_step_std': 0.4,
 'action_lower_bd': 0.2,
 'action_upper_bd': 1.8,
 'dot_std': 1.3,
 'img_size': 65,
 'border_wall_loc': 5,
 'fix_wall_batch_k': None,
 'fix_wall': False,
 'fix_door_location': 18,
 'fix_wall_location': 32,
 'exclude_wall_train': '',
 'exclude_door_train': '',
 'only_wall_val': '',
 'only_door_val': '',
 'wall_padding': 20,
 'door_padding': 10,
 'wall_width': 3,
 'door_space': 4,
 'num_train_layouts': -1,
 'cross_wall_rate': 0.35,
 'expert_cross_wall_rate': 0,
 'wall_bump_rate': 0.0,
 'dup_traj_rate': 0.0,
 'max_step': 1,
 'sample_length': 17,
 'n_steps': 91,
 'n_steps_reduce_factor': 1,
 'repeat_actions': 1,
 'size': 100000,
 'val_size': 10000,
 'train': True,
 'normalize': True,
 'batch_size': 128,
 'num_workers': 0,
 'pin_mem': False,
 'persistent_workers': False,
 'device': 'cpu'}

### `update_config_from_yaml`

In [4]:
from dataclasses import fields
from eb_jepa.datasets.two_rooms.wall_dataset import WallDatasetConfig

In [5]:
config_class = WallDatasetConfig
fields(config_class)

(Field(name='size',type=<class 'int'>,default=10000,default_factory=<dataclasses._MISSING_TYPE object at 0x108ee0230>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='val_size',type=<class 'int'>,default=10000,default_factory=<dataclasses._MISSING_TYPE object at 0x108ee0230>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='batch_size',type=<class 'int'>,default=128,default_factory=<dataclasses._MISSING_TYPE object at 0x108ee0230>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='dot_std',type=<class 'float'>,default=1.3,default_factory=<dataclasses._MISSING_TYPE object at 0x108ee0230>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='action_noise',type=<class 'float'>,default=0.2,default_factory=<dataclasses._MISSING_TYP

In [6]:
config_field_names = {f.name for f in fields(config_class)}

relevant_yaml_data = {
    key: value for key, value in base_config.items() if key in config_field_names
}

In [7]:
config = config_class(**relevant_yaml_data)  # notice the batch_size has overridden 
config

WallDatasetConfig(size=100000, val_size=10000, batch_size=128, dot_std=1.3, action_noise=1, action_angle_noise=0.2, action_step_mean=1.0, action_step_std=0.4, action_lower_bd=0.2, action_upper_bd=1.8, max_step=1, n_steps=91, img_size=65, train=True, device='cpu', repeat_actions=1, n_steps_reduce_factor=1, border_wall_loc=5, chunked_actions=False, normalize=True, fix_wall=False, fix_wall_batch_k=None, wall_padding=20, door_padding=10, wall_width=3, door_space=4, cross_wall_rate=0.35, expert_cross_wall_rate=0, wall_bump_rate=0.0, dup_traj_rate=0.0, expert_action_step_mean=0.9, expert_action_step_std=0, expert_action_lower_bd=0.9, expert_action_upper_bd=0.9, expert_traj_door_padding=2, exclude_wall_train='', exclude_door_train='', only_wall_val='', only_door_val='', fix_wall_location=32, fix_door_location=18, num_train_layouts=-1, image_based=True, sample_length=17)

# WallDataset

In [ ]:
from eb_jepa.datasets.two_rooms.wall_dataset import WallDataset

dset = WallDataset(config=config)   # from this loader, and val_loader are created. 
dset

# `init_data`

In [24]:
from pathlib import Path 

CFG_PATH = Path.cwd().parent.parent / "cfgs" / "train.yaml"

with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg["data"]["num_workers"] = 0 
cfg

{'logging': {'log_wandb': True,
  'wandb_group': None,
  'wandb_sweep': None,
  'log_every': 10,
  'exp_suffix': '26-07-20',
  'save_every_n_epochs': 1,
  'tqdm_silent': False},
 'meta': {'model_folder': None,
  'load_model': True,
  'enable_plan_eval': True,
  'eval_every_itr': -1,
  'light_eval_freq': 50,
  'seed': 1},
 'data': {'env_name': 'two_rooms',
  'batch_size': 384,
  'num_workers': 0,
  'pin_mem': True,
  'persistent_workers': True},
 'training': {'use_amp': True, 'dtype': 'bfloat16'},
 'model': {'compile': True,
  'dobs': 2,
  'henc': 32,
  'hpre': 32,
  'dstc': 32,
  'nsteps': 8,
  'encoder_architecture': 'impala',
  'regularizer': {'cov_coeff': 8,
   'std_coeff': 16,
   'sim_coeff_t': 12,
   'idm_coeff': 1,
   'first_t_only': False,
   'spatial_as_samples': False,
   'use_proj': False,
   'idm_after_proj': False,
   'sim_t_after_proj': False}},
 'optim': {'epochs': 12,
  'lr': 0.001,
  'grad_clip_enc': 2.0,
  'grad_clip_pred': 2.0,
  'weight_decay': 1e-05},
 'eval': {'pla

In [25]:
from eb_jepa.datasets.utils import init_data

env_name = "two_rooms"


loader, val_loader, data_config = init_data(env_name= env_name, cfg_data=dict(cfg["data"]))

In [26]:
data_config

WallDatasetConfig(size=100000, val_size=10000, batch_size=384, dot_std=1.3, action_noise=1, action_angle_noise=0.2, action_step_mean=1.0, action_step_std=0.4, action_lower_bd=0.2, action_upper_bd=1.8, max_step=1, n_steps=91, img_size=65, train=True, device='cpu', repeat_actions=1, n_steps_reduce_factor=1, border_wall_loc=5, chunked_actions=False, normalize=True, fix_wall=False, fix_wall_batch_k=None, wall_padding=20, door_padding=10, wall_width=3, door_space=4, cross_wall_rate=0.35, expert_cross_wall_rate=0, wall_bump_rate=0.0, dup_traj_rate=0.0, expert_action_step_mean=0.9, expert_action_step_std=0, expert_action_lower_bd=0.9, expert_action_upper_bd=0.9, expert_traj_door_padding=2, exclude_wall_train='', exclude_door_train='', only_wall_val='', only_door_val='', fix_wall_location=32, fix_door_location=18, num_train_layouts=-1, image_based=True, sample_length=17)

In [27]:
print(data_config.batch_size)
print(data_config.size) # train samples  
print(data_config.val_size)

steps_per_epoch = data_config.size // data_config.batch_size
print(steps_per_epoch)

print(data_config.img_size)

print(loader.dataset.normalizer)

384
100000
10000
260
65


In [28]:
batch = next(iter(loader))
batch

WallSample(states=tensor([[[[[-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           ...,
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705]],

          [[-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           ...,
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705],
           [-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705]],

          [[-0.0705, -0.0705, -0.0705,  ..., -0.0705, -0.0705, -0.0705

In [29]:
states, actions, loc, wall_x, door_y = batch 
print(states.shape)
print(actions.shape)
print(loc.shape)
print(wall_x.shape)
print(door_y.shape)

torch.Size([384, 2, 17, 65, 65])
torch.Size([384, 2, 17])
torch.Size([384, 2, 17])
torch.Size([384, 1])
torch.Size([384, 1])


In [30]:
for states, actions, loc, wall_x, door_y in  loader:
    print(states.shape)
    print(actions.shape)
    print(loc.shape)
    print(wall_x.shape)
    print(door_y.shape)
    break 

torch.Size([384, 2, 17, 65, 65])
torch.Size([384, 2, 17])
torch.Size([384, 2, 17])
torch.Size([384, 1])
torch.Size([384, 1])


In [31]:
for states, actions, loc, wall_x, door_y in  val_loader:
    print(states.shape)
    print(actions.shape)
    print(loc.shape)
    print(wall_x.shape)
    print(door_y.shape)
    break 

torch.Size([4, 2, 17, 65, 65])
torch.Size([4, 2, 17])
torch.Size([4, 2, 17])
torch.Size([4, 1])
torch.Size([4, 1])
